# Phase 1 — Data Understanding, Cleaning & Feature Engineering
### Project: Insurance Denial & Revenue Cycle Analysis (Tableau)
**Source:** Healthcare Fraud Detection Dataset (Kaggle)
https://www.kaggle.com/datasets/nudratabbas/healthcare-fraud-detection-dataset

This notebook consolidates Day 1 of the project plan:
1. Data Exploration
2. Data Cleaning
3. Feature Engineering
4. Final Validation & Export

**Before running:** Kernel → Restart & Run All, top to bottom, every time. Running cells
out of order is the single most common source of confusing bugs in this kind of notebook
(e.g., a column showing one value everywhere when it shouldn't) — always verify with a
clean linear run before trusting any output.


## 1. Data Exploration

In [1]:
import pandas as pd
import numpy as np

# ---- Load dataset ----
df = pd.read_csv("healthcare_fraud_detection.csv")

print("Shape (rows, columns):", df.shape)
print("\nColumn names and dtypes:")
print(df.dtypes)


Shape (rows, columns): (10000, 20)

Column names and dtypes:
Provider_ID                               object
Claim_ID                                  object
Patient_Age                                int64
Patient_Gender                            object
Diagnosis_Code                            object
Procedure_Code                             int64
Claim_Amount                             float64
Approved_Amount                          float64
Insurance_Type                            object
Claim_Submission_Date                     object
Days_Between_Service_and_Claim             int64
Number_of_Claims_Per_Provider_Monthly      int64
Provider_Specialty                        object
Patient_State                             object
Claim_Status                              object
Is_Fraud                                   int64
Length_of_Stay                             int64
Visit_Type                                object
Chronic_Condition_Flag                     int64
Prior_Vi

In [2]:
df.head(10)


,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0
5,P0241,C0000005,44,Female,E11.9,80053,220.99,174.05,Medicaid,2024-12-20,26,74,Orthopedics,CA,Approved,0,2,Emergency,0,1.0
6,P0196,C0000006,50,Male,E78.5,93000,458.60,372.31,Medicare,2022-02-20,2,84,Cardiology,OH,Approved,0,0,Outpatient,1,0.0
7,P0071,C0000007,1,Male,I10,85025,934.34,321.60,NaN,2022-07-13,6,114,NaN,PA,Pending,1,2,Emergency,0,NaN
8,P0097,C0000008,34,Female,M54.5,97110,478.11,412.48,Medicare,2021-05-25,24,65,Pulmonology,IL,Pending,0,5,Inpatient,1,4.0
9,P0125,C0000009,47,Male,K21.9,99213,426.16,335.80,Self-Pay,2022-05-29,5,56,Internal Medicine,TX,Approved,0,2,Emergency,0,4.0


In [3]:
# ---- Duplicate rows & missing values ----
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Claim_IDs:", df['Claim_ID'].duplicated().sum())
print("\nMissing values:\n", df.isna().sum())


Duplicate rows: 0
Duplicate Claim_IDs: 0

Missing values:
 Provider_ID                                0
Claim_ID                                   0
Patient_Age                                0
Patient_Gender                             0
Diagnosis_Code                             0
Procedure_Code                             0
Claim_Amount                               0
Approved_Amount                            0
Insurance_Type                           350
Claim_Submission_Date                      0
Days_Between_Service_and_Claim             0
Number_of_Claims_Per_Provider_Monthly      0
Provider_Specialty                       350
Patient_State                              0
Claim_Status                               0
Is_Fraud                                   0
Length_of_Stay                             0
Visit_Type                                 0
Chronic_Condition_Flag                     0
Prior_Visits_12m                         350
dtype: int64


In [4]:
# ---- Provider_ID <-> Provider_Specialty consistency check ----
# Confirms whether each provider has one consistent specialty, or conflicting/missing entries
provider_df = df[['Provider_ID', 'Provider_Specialty']].drop_duplicates()
provider_df.groupby('Provider_ID').size().value_counts()
provider_df

**Observations:**
- Rows Count - 10,000 rows, columns Count - 20.
- Full-row duplicates - 0; `Claim_ID` is unique per row
- Missing values: `Insurance_Type`, `Provider_Specialty`, `Prior_Visits_12m`


## 2. Data Cleaning

In [5]:
# ---- 2.1 Impute Provider_Specialty using other rows from the same Provider_ID ----
df['Provider_Specialty'] = (
    df.groupby('Provider_ID')['Provider_Specialty'].transform('first')
)

# Fallback for providers with NO valid specialty anywhere in their rows
remaining_null_specialty = df['Provider_Specialty'].isna().sum()
if remaining_null_specialty > 0:
    df['Provider_Specialty'] = df['Provider_Specialty'].fillna('Unknown Specialty')

print("Provider_Specialty after fill:", df['Provider_Specialty'].isna().sum())


Provider_Specialty after fill: 0


In [6]:
# ---- 2.2 Fix Insurance_Type missing values ----
df['Insurance_Type'] = df['Insurance_Type'].fillna('Unknown')

print("Insurance_Type distribution:")
print(df['Insurance_Type'].value_counts(dropna=False))


Insurance_Type distribution:
Insurance_Type
Medicare    2425
Medicaid    2419
Private     2405
Self-Pay    2401
Unknown      350
Name: count, dtype: int64


In [8]:
# ---- 2.3 Prior_Visits_12m — fill missing with 0 (assumes missing = no prior visits) ----
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(0)

print("Nulls in Prior_Visits_12m:", df['Prior_Visits_12m'].isna().sum())


Nulls in Prior_Visits_12m: 0


In [9]:
# ---- 2.4 Parse date columns ----
date_like_cols = [c for c in df.columns if "date" in c.lower()]
print("Date columns detected:", date_like_cols)

for date_col in date_like_cols:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    n_failed = df[date_col].isna().sum()
    if n_failed > 0:
        print(f"WARNING: {n_failed} unparseable dates in '{date_col}'")


Date columns detected: ['Claim_Submission_Date']


In [10]:
# ---- 2.5 Sanity-check amount columns (negatives, outliers) ----
amount_like_cols = [c for c in df.columns if any(k in c.lower() for k in ["amount", "amt"])]
print("Detected amount columns:", amount_like_cols)

for amt_col in amount_like_cols:
    if pd.api.types.is_numeric_dtype(df[amt_col]):
        n_negative = (df[amt_col] < 0).sum()
        print(f"{amt_col}: {n_negative} negative values | min={df[amt_col].min():.2f} | "
              f"max={df[amt_col].max():.2f} | median={df[amt_col].median():.2f}")


Detected amount columns: ['Claim_Amount', 'Approved_Amount']
Claim_Amount: 0 negative values | min=60.21 | max=6590.70 | median=461.23
Approved_Amount: 0 negative values | min=50.35 | max=4270.89 | median=388.37


In [12]:
# ---- 2.6 Sanity-check Patient_Age (catch obvious data errors) ----
age_cols = [c for c in df.columns if "age" in c.lower()]

for age_col in age_cols:
    print(f"{age_col}: min={df[age_col].min()}, max={df[age_col].max()}")
    bad_age = df[(df[age_col] < 0) | (df[age_col] > 110)]
    print(f"Rows with implausible age: {len(bad_age)}")


Patient_Age: min=1, max=95
Rows with implausible age: 0


In [13]:
# ---- 2.7 Standardize Claim_Status text (whitespace, casing) ----
df['Claim_Status'] = df['Claim_Status'].astype(str).str.strip().str.title()

print("Claim_Status value counts:")
print(df['Claim_Status'].value_counts(dropna=False))


Claim_Status value counts:
Claim_Status
Approved    7053
Rejected    1748
Pending     1199
Name: count, dtype: int64


In [14]:
# ---- 2.8 Full missing-values re-check after all cleaning steps ----
print("Missing values after cleaning:")
print(df.isna().sum())


Missing values after cleaning:
Provider_ID                              0
Claim_ID                                 0
Patient_Age                              0
Patient_Gender                           0
Diagnosis_Code                           0
Procedure_Code                           0
Claim_Amount                             0
Approved_Amount                          0
Insurance_Type                           0
Claim_Submission_Date                    0
Days_Between_Service_and_Claim           0
Number_of_Claims_Per_Provider_Monthly    0
Provider_Specialty                       0
Patient_State                            0
Claim_Status                             0
Is_Fraud                                 0
Length_of_Stay                           0
Visit_Type                               0
Chronic_Condition_Flag                   0
Prior_Visits_12m                         0
dtype: int64


## 3. Feature Engineering

In [16]:
# ---- 3.1 Submission Delay Bucket (NOT "aging" — see note below) ----
# This measures how long the PROVIDER took to submit the claim after service, not how long a claim has sat unresolved.

def submission_delay_bucket(days):
    if pd.isna(days):
        return "Unknown"
    elif days <= 7:
        return "0-7 days"
    elif days <= 15:
        return "8-15 days"
    elif days <= 30:
        return "16-30 days"
    else:
        return "30+ days"

df["Submission_Delay_Bucket"] = df["Days_Between_Service_and_Claim"].apply(submission_delay_bucket)
print(df["Submission_Delay_Bucket"].value_counts())


Submission_Delay_Bucket
16-30 days    4558
0-7 days      2827
8-15 days     2615
Name: count, dtype: int64


In [17]:
# ---- 3.2 Denial flag & denied dollar amount ----
df['is_denied'] = (df['Claim_Status'] == 'Rejected').astype(int)
df['denied_amount'] = np.where(df['is_denied'] == 1, df['Claim_Amount'], 0)

print("Denial rate:", round(df['is_denied'].mean() * 100, 2), "%")
print("Total denied $:", df['denied_amount'].sum())


Denial rate: 17.48 %
Total denied $: 1096633.3399999999


In [18]:
# ---- 3.3 Denial_Reason taxonomy (simulated — dataset has no native reason field) ----
DENIAL_REASONS = [
    "Prior Authorization Required",
    "Coding Error",
    "Eligibility Issue",
    "Duplicate Claim",
    "Missing Documentation",
    "Timely Filing Limit Exceeded",
    "Non-Covered Service",
]

rng = np.random.default_rng(seed=42)  # seeded for reproducibility across reruns

df['Denial_Reason'] = np.where(
    df['is_denied'] == 1,
    rng.choice(DENIAL_REASONS, size=len(df)),
    "N/A"
)

print("Denial_Reason distribution (Rejected claims only):")
print(df[df['is_denied'] == 1]['Denial_Reason'].value_counts())


Denial_Reason distribution (Rejected claims only):
Denial_Reason
Duplicate Claim                 271
Non-Covered Service             256
Coding Error                    254
Prior Authorization Required    250
Eligibility Issue               247
Timely Filing Limit Exceeded    237
Missing Documentation           233
Name: count, dtype: int64


In [19]:
# ---- 3.4 Collection Rate % ----
df['Collection_Rate_pct'] = (df['Approved_Amount'] / df['Claim_Amount']) * 100

# Sanity check: any claims paying out MORE than billed?
over_100 = df[df['Collection_Rate_pct'] > 100]
print(f"Claims with Collection_Rate_pct > 100%: {len(over_100)}")
if len(over_100) > 0:
    print(over_100[['Claim_ID', 'Claim_Amount', 'Approved_Amount', 'Collection_Rate_pct']].head())


Claims with Collection_Rate_pct > 100%: 0


In [21]:
# ---- 3.5 Collection Rate by Claim_Status — check whether Pending claims skew the metric ----
print(df.groupby('Claim_Status')['Collection_Rate_pct'].describe())

# exclude Pending claims from any finalized "Net Collection Rate" KPI

finalized_df = df[df['Claim_Status'] != 'Pending']
net_collection_rate = (finalized_df['Approved_Amount'].sum() / finalized_df['Claim_Amount'].sum()) * 100
print(f"\nNet Collection Rate (Approved + Rejected only, excludes Pending): {net_collection_rate:.2f}%")


               count       mean        std        min        25%        50%  \
Claim_Status                                                                  
Approved      7053.0  86.944323   8.805897  30.048440  80.991379  87.419475   
Pending       1199.0  79.999203  16.584378  30.268038  76.005522  83.927410   
Rejected      1748.0  79.884448  16.797915  30.171633  76.113651  83.786345   

                    75%        max  
Claim_Status                        
Approved      93.771573  99.994540  
Pending       91.779925  99.896014  
Rejected      91.849036  99.991845  

Net Collection Rate (Approved + Rejected only, excludes Pending): 84.12%


## 4. Final Validation & Export

In [22]:
key_cols = ['Claim_ID', 'Claim_Status', 'is_denied', 'denied_amount',
            'Collection_Rate_pct', 'Denial_Reason', 'Submission_Delay_Bucket',
            'Insurance_Type', 'Provider_Specialty', 'Prior_Visits_12m']

print("Final null check on key output columns:")
print(df[key_cols].isna().sum())

print(f"\nFinal shape: {df.shape}")


Final null check on key output columns:
Claim_ID                   0
Claim_Status               0
is_denied                  0
denied_amount              0
Collection_Rate_pct        0
Denial_Reason              0
Submission_Delay_Bucket    0
Insurance_Type             0
Provider_Specialty         0
Prior_Visits_12m           0
dtype: int64

Final shape: (10000, 25)


In [23]:
OUTPUT_PATH = "cleaned_claims_data.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUTPUT_PATH}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")


Saved cleaned dataset to cleaned_claims_data.csv
Rows: 10000, Columns: 25


In [26]:
OUTPUT_PATH = "cleaned_claims_data.xlsx"

df.to_excel(OUTPUT_PATH, index=False)

print(f"Saved cleaned dataset to {OUTPUT_PATH}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")


Saved cleaned dataset to cleaned_claims_data.xlsx
Rows: 10000, Columns: 25


In [25]:
!pip install openpyxl


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
